# 03 · Unsupervised and semi-supervised learning (guided)

**Style A: syntax gaps.** 8 gaps (`# TODO(gN)` + `____`), each followed by a check.
Solutions: `../solutions/guided/03_unsupervised.md`.

**Traces to** `notes/ioai-task-patterns.md` P7. The reference solution to T25-ANTIQUE used
`SpectralClustering`, then a majority label per cluster, then an SVC. T24-O-HYPER allowed unsupervised
feature extractors such as PCA. Syllabus: K-means, PCA (Both); t-SNE/UMAP, DBSCAN/hierarchical/spectral (Practice).

Runtime: about 1 minute on CPU (t-SNE is the slow part).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN, SpectralClustering, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.semi_supervised import LabelSpreading

digits = load_digits()
X = digits.data / 16.0      # pixels are already on one scale; StandardScaler would blow up the near-empty border pixels
y = digits.target
print(X.shape)

## 1. PCA: how many components keep 90 % of the variance?

In [ ]:
pca = PCA().fit(X)
cum = np.cumsum(pca.explained_variance_ratio_)
# TODO(g1): smallest k with cumulative explained variance >= 0.90
k = ____
Z = PCA(n_components=k, random_state=0).fit_transform(X)
print(k, Z.shape)

In [ ]:
assert cum[k - 1] >= 0.90 and cum[k - 2] < 0.90, "off by one?"
print("g1 ok")

## 2. K-means and a permutation-invariant score

Cluster ids are arbitrary: cluster 3 is not digit 3. Compare clusterings with **ARI**, which ignores label names.

In [ ]:
# TODO(g2): 10 clusters, 10 random restarts (k-means only finds a local optimum), random_state=0
km = ____.fit(Z)
# TODO(g3): similarity between true digits y and the cluster labels, invariant to renaming clusters
ari = ____
print(f"ARI = {ari:.3f}, silhouette = {silhouette_score(Z, km.labels_):.3f}")

In [ ]:
assert km.n_init == 10 and len(np.unique(km.labels_)) == 10
renamed = (km.labels_ + 3) % 10
assert np.isclose(ari, adjusted_rand_score(y, renamed)), "the score must not change when clusters are renamed"
assert ari > 0.5
print("g2, g3 ok")

## 3. From clusters to labels with a few known labels

T25-ANTIQUE's trick: give each cluster the **majority label of the few labelled points inside it**.

In [ ]:
rng = np.random.default_rng(0)
known = rng.random(len(y)) < 0.05                     # we only know 5 % of the labels
mapping = {}
for c in range(10):
    labels_in_c = y[known & (km.labels_ == c)]
    # TODO(g4): most frequent label among labels_in_c (fall back to -1 if the cluster has no known label)
    mapping[c] = ____ if len(labels_in_c) else -1
pred = np.array([mapping[c] for c in km.labels_])
print("accuracy on the unknown 95 %:", round((pred[~known] == y[~known]).mean(), 3))

In [ ]:
c0 = y[known & (km.labels_ == 0)]
assert mapping[0] == (np.bincount(c0).argmax() if len(c0) else -1)
assert (pred[~known] == y[~known]).mean() > 0.6
print("g4 ok")

## 4. t-SNE for a 2-D picture

In [ ]:
# TODO(g5): t-SNE has no separate .transform(): fit and embed in one call
emb = TSNE(n_components=2, init="pca", perplexity=30, random_state=0).____
plt.figure(figsize=(5, 4)); plt.scatter(emb[:, 0], emb[:, 1], c=y, s=4, cmap="tab10"); plt.colorbar(); plt.show()

In [ ]:
assert emb.shape == (len(X), 2)
print("g5 ok")

## 5. Density- and graph-based clustering on two moons

K-means cuts the moons with a straight line. DBSCAN and spectral clustering follow the shape.
DBSCAN labels outliers `-1`.

In [ ]:
Xm, ym = make_moons(n_samples=600, noise=0.06, random_state=0)
db = DBSCAN(eps=0.15, min_samples=5).fit(Xm)
# TODO(g6): how many points did DBSCAN mark as noise?
n_noise = ____
sc = SpectralClustering(n_clusters=2, affinity="nearest_neighbors", n_neighbors=10, random_state=0).fit_predict(Xm)
ag = AgglomerativeClustering(n_clusters=2, linkage="single").fit_predict(Xm)
km2 = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(Xm)
for name, lab in [("kmeans", km2), ("dbscan", db.labels_), ("spectral", sc), ("single-link", ag)]:
    print(f"{name:12s} ARI {adjusted_rand_score(ym, lab):.3f}")
print("noise points:", n_noise)

In [ ]:
assert n_noise == list(db.labels_).count(-1)
print("g6 ok")

## 6. Label spreading: sklearn's semi-supervised convention

In sklearn's semi-supervised estimators, **unlabelled samples must be marked `-1`**. T25-ANTIQUE used `0`
for "unknown", so remap before fitting.

In [ ]:
y_semi = y.copy()
# TODO(g7): mark every sample we do NOT know (~known) as unlabelled, sklearn style
y_semi[~known] = ____
# TODO(g8): k-NN graph label spreading with 10 neighbours
ls = ____.fit(Z, y_semi)
acc = (ls.transduction_[~known] == y[~known]).mean()
print(f"label spreading accuracy on the unknown 95 %: {acc:.3f}")

In [ ]:
assert (y_semi == -1).sum() == (~known).sum()
assert acc > 0.9, acc
print("g7, g8 ok, all gaps done")